# Previsao de Precos de Carros no Brasil - Tabela FIPE (2021-2023)

**Objetivo:** Construir e comparar multiplos modelos de Machine Learning para prever o preco medio de veiculos com base na Tabela FIPE, selecionar o melhor modelo e salva-lo para uso em producao.

---

## Dataset
- **Fonte:** Arquivo local fipe_cars.csv
- **Periodo:** Janeiro de 2021 a Janeiro de 2023
- **Volume:** ~600 mil registros de carros
- **Target:** `avg_price_brl` - Preco medio do veiculo em Reais (R$)

## Estrutura do Notebook
1. Instalacao & Imports
2. Carregamento & Inspecao dos Dados
3. Analise Exploratoria (EDA)
4. Pre-processamento & Feature Engineering
5. Divisao Treino / Validacao / Teste
6. Treinamento de Multiplos Modelos
7. Comparacao de Performance
8. Tuning do Melhor Modelo
9. Analise de Importancia de Features
10. Salvamento do Melhor Modelo

## 1. Instalacao & Imports

In [ ]:
# Instalar dependencias (caso necessario)
# !pip install kaggle xgboost lightgbm scikit-learn pandas numpy matplotlib seaborn joblib -q

# Manipulacao de Dados
import pandas as pd
import numpy as np

# Visualizacao
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Pre-processamento
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline

# Modelos de ML
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    ExtraTreesRegressor,
    AdaBoostRegressor,
)
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
import xgboost as xgb
import lightgbm as lgb

# Metricas
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    mean_absolute_percentage_error,
)

# Persistencia do Modelo
import joblib
import os
import warnings
warnings.filterwarnings("ignore")

# Configuracoes de visualizacao
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("husl")
pd.set_option("display.float_format", "{:.2f}".format)
pd.set_option("display.max_columns", 50)

print("Todos os imports realizados com sucesso!")
print(f"   XGBoost  : {xgb.__version__}")
print(f"   LightGBM : {lgb.__version__}")

## 2. Carregamento & Inspecao dos Dados

O dataset e carregado do arquivo local fipe_cars.csv localizado na pasta /data.

In [ ]:
# Carregar dados reais do arquivo local
data_path = "data/fipe_cars.csv"
df_raw = pd.read_csv(data_path)

print(f"Dataset carregado - Shape: {df_raw.shape}")
print(f"Periodo de referencia: {df_raw.year_of_reference.min()}"
      f"/{df_raw.month_of_reference.min()} -> "
      f"{df_raw.year_of_reference.max()}/{df_raw.month_of_reference.max()}")
df_raw.head()

## 3. Analise Exploratoria de Dados (EDA)

Antes de modelar, e essencial entender a distribuicao dos dados, outliers, correlacoes e padroes relevantes.

In [ ]:
# Informacoes gerais
print("=" * 60)
print("INFORMACOES DO DATASET")
print("=" * 60)
print(df_raw.info())

print("\n" + "=" * 60)
print("ESTATISTICAS DESCRITIVAS - VARIÁVEIS NUMERICAS")
print("=" * 60)
df_raw.describe().T

In [ ]:
# Valores nulos
nulls = df_raw.isnull().sum()
print("Valores nulos por coluna:")
print(nulls[nulls > 0] if nulls.any() else "  Nenhum valor nulo encontrado!")

# Duplicatas
print(f"\nDuplicatas: {df_raw.duplicated().sum()}")

In [ ]:
# DISTRIBUICAO DO PRECO (TARGET)
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Histograma original
axes[0].hist(df_raw["avg_price_brl"], bins=80, color="#2196F3", edgecolor="white", alpha=0.85)
axes[0].set_title("Distribuicao do Preco Medio (R$)", fontsize=14, fontweight="bold")
axes[0].set_xlabel("Preco (R$)")
axes[0].set_ylabel("Frequencia")
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"R$ {x/1e3:.0f}k"))

# Distribuicao log (mais util para regressao)
axes[1].hist(np.log1p(df_raw["avg_price_brl"]), bins=80,
             color="#4CAF50", edgecolor="white", alpha=0.85)
axes[1].set_title("Distribuicao Log(Preco + 1)", fontsize=14, fontweight="bold")
axes[1].set_xlabel("log(Preco + 1)")
axes[1].set_ylabel("Frequencia")

plt.suptitle("Analise do Target: avg_price_brl", fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

skewness = df_raw["avg_price_brl"].skew()
print(f"\nAssimetria (skewness) original : {skewness:.3f}")
print(f"   Assimetria apos log1p            : {np.log1p(df_raw['avg_price_brl']).skew():.3f}")
print("   Usaremos log-transform para melhorar os modelos lineares.")

In [ ]:
# PRECO MEDIO POR MARCA (Top 15)
avg_by_brand = (df_raw.groupby("brand")["avg_price_brl"]
                .median()
                .sort_values(ascending=False)
                .head(15))

fig, ax = plt.subplots(figsize=(14, 6))
bars = ax.barh(avg_by_brand.index, avg_by_brand.values, color=sns.color_palette("husl", 15))
ax.set_xlabel("Preco Mediano (R$)")
ax.set_title("Preco Mediano por Marca (Top 15)", fontsize=14, fontweight="bold")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"R$ {x/1e3:.0f}k"))
for bar, val in zip(bars, avg_by_brand.values):
    ax.text(val + 2000, bar.get_y() + bar.get_height()/2,
            f"R$ {val/1e3:.0f}k", va="center", fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# PRECO POR COMBUSTIVEL E CAMBIO
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Combustivel
fuel_order = df_raw.groupby("fuel")["avg_price_brl"].median().sort_values(ascending=False).index
sns.boxplot(data=df_raw, x="fuel", y="avg_price_brl", order=fuel_order, ax=axes[0],
            palette="Set2", showfliers=False)
axes[0].set_title("Preco por Tipo de Combustivel", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Combustivel")
axes[0].set_ylabel("Preco (R$)")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"R$ {x/1e3:.0f}k"))
axes[0].tick_params(axis="x", rotation=15)

# Cambio
gear_order = df_raw.groupby("gear")["avg_price_brl"].median().sort_values(ascending=False).index
sns.boxplot(data=df_raw, x="gear", y="avg_price_brl", order=gear_order, ax=axes[1],
            palette="Set3", showfliers=False)
axes[1].set_title("Preco por Tipo de Cambio", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Cambio")
axes[1].set_ylabel("Preco (R$)")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"R$ {x/1e3:.0f}k"))
axes[1].tick_params(axis="x", rotation=15)

plt.suptitle("Distribuicao de Precos por Categoria", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# PRECO x ANO DO MODELO (depreciacao)
avg_by_year = df_raw.groupby("year_model")["avg_price_brl"].median().reset_index()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(avg_by_year["year_model"], avg_by_year["avg_price_brl"],
        marker="o", linewidth=2, color="#E91E63", markersize=5)
ax.fill_between(avg_by_year["year_model"], avg_by_year["avg_price_brl"],
                alpha=0.15, color="#E91E63")
ax.set_title("Preco Mediano por Ano do Modelo", fontsize=14, fontweight="bold")
ax.set_xlabel("Ano do Modelo")
ax.set_ylabel("Preco Mediano (R$)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"R$ {x/1e3:.0f}k"))
plt.tight_layout()
plt.show()

print("Insight: Carros mais novos sao significativamente mais caros.")
print("   A depreciacao e um dos fatores mais fortes na previsao do preco.")

In [ ]:
# PRECO x TAMANHO DO MOTOR
avg_by_engine = df_raw.groupby("engine_size")["avg_price_brl"].median().reset_index()

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(avg_by_engine["engine_size"].astype(str), avg_by_engine["avg_price_brl"],
       color=sns.color_palette("viridis", len(avg_by_engine)))
ax.set_title("Preco Mediano por Cilindrada do Motor", fontsize=14, fontweight="bold")
ax.set_xlabel("Cilindrada (L)")
ax.set_ylabel("Preco Mediano (R$)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"R$ {x/1e3:.0f}k"))
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

## 4. Pre-processamento & Feature Engineering

Etapas essenciais antes do treinamento:
- Remocao de colunas irrelevantes (IDs, autenticacoes)
- Criacao de novas features (idade do carro, log do preco)
- Encoding de variaveis categoricas
- Tratamento de outliers extremos

In [ ]:
df = df_raw.copy()

# 4.1 Remover colunas de identificacao
# fipe_code e authentication sao IDs unicos - nao tem poder preditivo
cols_to_drop = ["fipe_code", "authentication", "model"]
df.drop(columns=cols_to_drop, inplace=True)
print(f"Colunas removidas: {cols_to_drop}")

# 4.2 Feature Engineering
df["car_age"] = 2023 - df["year_model"]               # Idade do carro em 2023
df["is_luxury"] = df["brand"].isin(
    ["BMW", "Mercedes-Benz", "Audi", "Porsche", "Land Rover"]
).astype(int)                                          # Flag de luxo

print("Novas features criadas:")
print("   car_age   - Idade do veiculo (anos)")
print("   is_luxury - Flag: marca premium (1/0)")

# 4.3 Target: log-transform
# A distribuicao do preco tem assimetria positiva forte.
# O log1p reduz isso e melhora a performance de modelos lineares.
df["log_price"] = np.log1p(df["avg_price_brl"])
print("\nTarget log_price = log1p(avg_price_brl)")

# 4.4 Outliers extremos
# Removemos precos abaixo do percentil 1 e acima do percentil 99
p1, p99 = df["avg_price_brl"].quantile([0.01, 0.99])
before = len(df)
df = df[(df["avg_price_brl"] >= p1) & (df["avg_price_brl"] <= p99)]
print(f"\nOutliers removidos: {before - len(df)} registros "
      f"({(before-len(df))/before*100:.1f}%)")
print(f"   Faixa de preco mantida: R$ {p1:,.0f} -> R$ {p99:,.0f}")

# 4.5 Encoding de categoricas - TODAS as colunas categóricas
# Label Encoding para arvores de decisao e boosting
le_dict = {}
cat_cols = ["brand", "fuel", "gear", "month_of_reference"]
for col in cat_cols:
    le = LabelEncoder()
    df[f"{col}_enc"] = le.fit_transform(df[col])
    le_dict[col] = le
    print(f"   {col}: {len(le.classes_)} classes")

# Remover colunas categóricas originais (agora temos as versões _enc)
df.drop(columns=cat_cols, inplace=True)

print(f"\nEncoding realizado. Colunas disponíveis: {list(df.columns)}")
print(f"Shape final: {df.shape}")

# Verificar se todas as colunas são numéricas
print("\nVerificacao de tipos:")
print(df.dtypes)

In [ ]:
# MAPA DE CORRELACAO
numeric_df = df.select_dtypes(include=[np.number])
corr = numeric_df.corr()

fig, ax = plt.subplots(figsize=(12, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, square=True, linewidths=0.5, ax=ax,
            annot_kws={"size": 9})
ax.set_title("Mapa de Correlacao das Features", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print("\nTop correlacoes com log_price:")
print(corr["log_price"].sort_values(ascending=False).drop("log_price").to_string())

## 5. Divisao Treino / Validacao / Teste

Dividimos os dados em tres conjuntos:
- **Treino (70%):** para ajustar os modelos
- **Validacao (15%):** para comparar modelos e tunar hiperparametros
- **Teste (15%):** avaliacao final - **visto apenas uma vez!**

In [ ]:
# Separar features e target
# IMPORTANTE: Usar apenas as colunas codificadas (_enc), nunca as originais categóricas
FEATURES = [
    'year_of_reference',
    'month_of_reference_enc',
    'engine_size',
    'year_model',
    'car_age',
    'is_luxury',
    'brand_enc',
    'fuel_enc',
    'gear_enc'
]
TARGET_LOG = "log_price"      # usamos log-price no treino
TARGET_RAW = "avg_price_brl"  # convertemos de volta para R$ na avaliacao

X = df[FEATURES]
y_log = df[TARGET_LOG]
y_raw = df[TARGET_RAW]

print(f"Features utilizadas ({len(FEATURES)}):")
for f in FEATURES:
    print(f"  - {f}")

print(f"\nVerificacao: Todas as features sao numericas?")
print(X.dtypes)

# Split 70 / 15 / 15
X_train, X_temp, y_log_train, y_log_temp = train_test_split(
    X, y_log, test_size=0.30, random_state=42)
X_val, X_test, y_log_val, y_log_test = train_test_split(
    X_temp, y_log_temp, test_size=0.50, random_state=42)

# Guardar precos reais correspondentes
y_raw_train = np.expm1(y_log_train)
y_raw_val   = np.expm1(y_log_val)
y_raw_test  = np.expm1(y_log_test)

print(f"\nDivisao dos dados:")
print(f"   Treino     : {X_train.shape[0]:>7,} amostras ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"   Validacao  : {X_val.shape[0]:>7,} amostras ({X_val.shape[0]/len(X)*100:.1f}%)")
print(f"   Teste      : {X_test.shape[0]:>7,} amostras ({X_test.shape[0]/len(X)*100:.1f}%)")

## 6. Treinamento de Multiplos Modelos

Testado com **10 modelos diferentes** cobrindo diferentes familias de algoritmos:
- **Lineares:** Linear Regression, Ridge, Lasso, ElasticNet
- **Baseados em Arvores:** Decision Tree, Random Forest, Extra Trees
- **Boosting:** Gradient Boosting, XGBoost, LightGBM

Todos os modelos sao avaliados no conjunto de **validacao** para selecao justa.

In [ ]:
import time

# Definicao dos modelos a comparar
models = {
    "Linear Regression": LinearRegression(),
    "Ridge":             Ridge(alpha=1.0, random_state=42),
    "Lasso":             Lasso(alpha=0.001, max_iter=5000, random_state=42),
    "ElasticNet":        ElasticNet(alpha=0.001, l1_ratio=0.5, max_iter=5000, random_state=42),
    "Decision Tree":     DecisionTreeRegressor(max_depth=10, random_state=42),
    "Random Forest":     RandomForestRegressor(n_estimators=150, max_depth=12,
                                               n_jobs=-1, random_state=42),
    "Extra Trees":       ExtraTreesRegressor(n_estimators=150, max_depth=12,
                                             n_jobs=-1, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=150, learning_rate=0.1,
                                                    max_depth=5, random_state=42),
    "XGBoost":           xgb.XGBRegressor(n_estimators=200, learning_rate=0.08,
                                           max_depth=6, subsample=0.8,
                                           colsample_bytree=0.8, n_jobs=-1,
                                           random_state=42, verbosity=0),
    "LightGBM":          lgb.LGBMRegressor(n_estimators=200, learning_rate=0.08,
                                            max_depth=6, subsample=0.8,
                                            colsample_bytree=0.8, n_jobs=-1,
                                            random_state=42, verbose=-1),
}

# Treinamento e avaliacao
results = []

for name, model in models.items():
    print(f"Treinando: {name:<25}", end="", flush=True)
    t0 = time.time()

    model.fit(X_train, y_log_train)
    elapsed = time.time() - t0

    # Previsoes em R$ (convertendo de volta do espaco log)
    pred_val_log = model.predict(X_val)
    pred_val_brl = np.expm1(pred_val_log)

    mae   = mean_absolute_error(y_raw_val, pred_val_brl)
    rmse  = np.sqrt(mean_squared_error(y_raw_val, pred_val_brl))
    mape  = mean_absolute_percentage_error(y_raw_val, pred_val_brl) * 100
    r2    = r2_score(y_raw_val, pred_val_brl)

    results.append({
        "Modelo":      name,
        "MAE (R$)":    mae,
        "RMSE (R$)":   rmse,
        "MAPE (%)":    mape,
        "R² Score":    r2,
        "Tempo (s)":   elapsed,
    })
    print(f"  OK  R²={r2:.4f} | MAE=R$ {mae:>10,.0f} | Tempo={elapsed:.1f}s")

results_df = pd.DataFrame(results).sort_values("R² Score", ascending=False).reset_index(drop=True)
print("\nTreinamento concluido!")
results_df

## 7. Comparacao de Performance dos Modelos

Visualizamos as principais metricas de todos os modelos para escolher o vencedor.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 12))

metrics = [
    ("R² Score",  True,  "#4CAF50", "R²"),
    ("MAE (R$)",  False, "#F44336", "MAE"),
    ("RMSE (R$)", False, "#FF9800", "RMSE"),
    ("MAPE (%)",  False, "#9C27B0", "MAPE"),
]

for ax, (metric, higher_better, color, label) in zip(axes.flat, metrics):
    sorted_df = results_df.sort_values(metric, ascending=not higher_better)
    bars = ax.barh(sorted_df["Modelo"], sorted_df[metric], color=color, alpha=0.85)
    ax.set_title(f"{label} - {'maior e melhor' if higher_better else 'menor e melhor'}",
                 fontsize=12, fontweight="bold")
    ax.set_xlabel(metric)

    # Destaque do melhor
    best_idx = 0 if higher_better else -1
    bars[best_idx].set_edgecolor("black")
    bars[best_idx].set_linewidth(2.5)
    bars[best_idx].set_alpha(1.0)

    # Rotulos
    for bar, val in zip(bars, sorted_df[metric]):
        fmt = f"{val:.4f}" if metric == "R² Score" else (
              f"{val:.1f}%" if "%" in metric else f"R$ {val:,.0f}")
        ax.text(bar.get_width() * 0.02, bar.get_y() + bar.get_height()/2,
                fmt, va="center", ha="left", fontsize=8, fontweight="bold")

    if metric in ["MAE (R$)", "RMSE (R$)"]:
        ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"R$ {x/1e3:.0f}k"))

plt.suptitle("Comparacao de Performance - Conjunto de Validacao",
             fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()

print("\nRanking Final (por R²):")
print(results_df[["Modelo", "R² Score", "MAE (R$)", "MAPE (%)"]].to_string(index=False))

## 8. Selecao & Avaliacao Final do Melhor Modelo

O modelo com maior **R²** no conjunto de validacao e selecionado automaticamente.
A avaliacao final e feita **uma unica vez** no conjunto de **teste** para garantir imparcialidade.

In [ ]:
# Identificar e recuperar o melhor modelo
best_name  = results_df.iloc[0]["Modelo"]
best_model = models[best_name]

print(f"MELHOR MODELO: {best_name}")
print(f"   R² Validacao : {results_df.iloc[0]['R² Score']:.4f}")
print(f"   MAE Validacao: R$ {results_df.iloc[0]['MAE (R$)']:,.0f}")
print(f"   MAPE         : {results_df.iloc[0]['MAPE (%)']:.2f}%")

# Avaliacao final no conjunto de TESTE (invisivel durante toda a fase de tuning)
pred_test_log = best_model.predict(X_test)
pred_test_brl = np.expm1(pred_test_log)

mae_test  = mean_absolute_error(y_raw_test, pred_test_brl)
rmse_test = np.sqrt(mean_squared_error(y_raw_test, pred_test_brl))
mape_test = mean_absolute_percentage_error(y_raw_test, pred_test_brl) * 100
r2_test   = r2_score(y_raw_test, pred_test_brl)

print("\n" + "="*55)
print(f"  AVALIACAO FINAL - CONJUNTO DE TESTE")
print("="*55)
print(f"  R² Score : {r2_test:.4f}")
print(f"  MAE      : R$ {mae_test:>12,.0f}")
print(f"  RMSE     : R$ {rmse_test:>12,.0f}")
print(f"  MAPE     : {mape_test:.2f}%")
print("="*55)
print(f"\nInterpretacao:")
print(f"   Em media, o modelo erra ±{mape_test:.1f}% no preco do veiculo.")
print(f"   Ou seja, para um carro de R$ 100.000, o erro medio e ~ R$ {100_000*mape_test/100:,.0f}.")

In [ ]:
# GRAFICO: PREVISTO x REAL
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Amostra para o scatter (evitar sobrecarga visual)
idx = np.random.choice(len(y_raw_test), min(3000, len(y_raw_test)), replace=False)
y_sample   = np.array(y_raw_test)[idx]
pred_sample = pred_test_brl[idx]

axes[0].scatter(y_sample, pred_sample, alpha=0.3, s=10, color="#2196F3")
lim = [min(y_sample.min(), pred_sample.min()),
       max(y_sample.max(), pred_sample.max())]
axes[0].plot(lim, lim, "r--", linewidth=1.5, label="Perfeito (y=x)")
axes[0].set_xlabel("Preco Real (R$)")
axes[0].set_ylabel("Preco Previsto (R$)")
axes[0].set_title("Previsto x Real", fontsize=13, fontweight="bold")
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"R$ {x/1e3:.0f}k"))
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"R$ {x/1e3:.0f}k"))
axes[0].legend()

# Residuos
residuals = y_sample - pred_sample
axes[1].scatter(pred_sample, residuals, alpha=0.3, s=10, color="#FF5722")
axes[1].axhline(0, color="red", linestyle="--", linewidth=1.5)
axes[1].set_xlabel("Preco Previsto (R$)")
axes[1].set_ylabel("Residuo (R$)")
axes[1].set_title("Residuos vs. Previsto", fontsize=13, fontweight="bold")
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"R$ {x/1e3:.0f}k"))
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"R$ {x/1e3:.0f}k"))

plt.suptitle(f"Analise do Melhor Modelo: {best_name}", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.show()

print("Residuos bem distribuidos ao redor de zero indicam que o modelo")
print("   nao tem vies sistemático significativo.")

## 9. Importancia das Features

Entender quais variaveis mais influenciam a previsao de preco.

In [ ]:
# Feature importance (disponivel em modelos baseados em arvores)
if hasattr(best_model, "feature_importances_"):
    importances = pd.Series(best_model.feature_importances_, index=FEATURES)
    importances = importances.sort_values(ascending=False)

    fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.barh(importances.index[::-1], importances.values[::-1],
                   color=sns.color_palette("viridis", len(importances)))
    ax.set_title(f"Importancia das Features - {best_name}",
                 fontsize=14, fontweight="bold")
    ax.set_xlabel("Importancia Relativa")
    for bar, val in zip(bars, importances.values[::-1]):
        ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
                f"{val:.3f}", va="center", fontsize=9)
    plt.tight_layout()
    plt.show()

    print("\nTop features e suas importancias:")
    for feat, imp in importances.items():
        print(f"   {feat:<20} : {imp:.4f} ({imp*100:.1f}%)")
else:
    # Para modelos lineares: coeficientes normalizados
    if hasattr(best_model, "coef_"):
        coefs = pd.Series(np.abs(best_model.coef_), index=FEATURES).sort_values(ascending=False)
        fig, ax = plt.subplots(figsize=(10, 5))
        ax.barh(coefs.index[::-1], coefs.values[::-1], color="#9C27B0", alpha=0.85)
        ax.set_title(f"|Coeficientes| - {best_name}", fontsize=14, fontweight="bold")
        ax.set_xlabel("|Coeficiente| (espaco log)")
        plt.tight_layout()
        plt.show()

## 10. Salvamento do Melhor Modelo

O modelo, encoders e metadados sao salvos em disco para uso em producao.

In [ ]:
import json

# Criar pasta de saida
os.makedirs("modelo_fipe", exist_ok=True)

# 1. Modelo principal
model_path = "modelo_fipe/melhor_modelo.pkl"
joblib.dump(best_model, model_path, compress=3)
print(f"Modelo salvo em: {model_path}")
print(f"   Tamanho: {os.path.getsize(model_path)/1024:.1f} KB")

# 2. Label encoders
encoders_path = "modelo_fipe/label_encoders.pkl"
joblib.dump(le_dict, encoders_path)
print(f"Encoders salvos em: {encoders_path}")

# 3. Lista de features
features_path = "modelo_fipe/features.pkl"
joblib.dump(FEATURES, features_path)
print(f"Features salvas em: {features_path}")

# 4. Metadados do modelo
metadata = {
    "model_name":    best_name,
    "features":      FEATURES,
    "target":        "avg_price_brl (via log1p transform)",
    "train_samples": int(len(X_train)),
    "val_samples":   int(len(X_val)),
    "test_samples":  int(len(X_test)),
    "metrics_test": {
        "r2":    round(float(r2_test),   4),
        "mae":   round(float(mae_test),  2),
        "rmse":  round(float(rmse_test), 2),
        "mape":  round(float(mape_test), 2),
    },
    "trained_on":    "FIPE Dataset (arquivo local: data/fipe_cars.csv)",
    "created_at":    pd.Timestamp.now().isoformat(),
}
meta_path = "modelo_fipe/metadata.json"
with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)
print(f"Metadados salvos em: {meta_path}")

print("\nConteudo da pasta modelo_fipe/:")
for f in os.listdir("modelo_fipe"):
    path = f"modelo_fipe/{f}"
    print(f"   {f:<35} ({os.path.getsize(path)/1024:.1f} KB)")

In [ ]:
# DEMONSTRACAO: Como usar o modelo salvo
print("=" * 60)
print("  DEMONSTRACAO - Usando o modelo em producao")
print("=" * 60)

# Carregar modelo e artefatos
modelo_carregado = joblib.load("modelo_fipe/melhor_modelo.pkl")
encoders         = joblib.load("modelo_fipe/label_encoders.pkl")

# Carregar as features corretas salvas
features_list    = joblib.load("modelo_fipe/features.pkl")

# Exemplo de novo carro para previsao
novo_carro = {
    "year_of_reference":  2023,
    "year_model":         2020,
    "engine_size":        1.6,
    "car_age":            3,        # 2023 - 2020
    "is_luxury":          0,
    "brand_enc":          encoders["brand"].transform(["VW - VolksWagen"])[0],
    "fuel_enc":           encoders["fuel"].transform(["Gasoline"])[0],
    "gear_enc":           encoders["gear"].transform(["manual"])[0],
    "month_of_reference_enc": encoders["month_of_reference"].transform(["January"])[0],
}

# Criar DataFrame usando a ordem correta de features
X_novo = pd.DataFrame([novo_carro])[features_list]
pred_log  = modelo_carregado.predict(X_novo)[0]
pred_brl  = np.expm1(pred_log)

print("\nVeiculo de exemplo:")
print(f"   Marca      : Volkswagen")
print(f"   Ano Modelo : 2020")
print(f"   Combustivel: Gasolina")
print(f"   Cambio     : Manual")
print(f"   Motor      : 1.6L")
print(f"\nPreco previsto: R$ {pred_brl:,.2f}")
print(f"\nPipeline de inferencia funcionando corretamente!")

## Conclusao

### Resumo dos Resultados

| Etapa                    | Detalhe |
|--------------------------|---------|
| Dataset                  | FIPE - Precos Medios de Carros no Brasil |
| Registros usados         | ~600.000 |
| Features selecionadas    | 9 variaveis (incluindo 2 criadas) |
| Modelos testados         | 10 algoritmos diferentes |
| Melhor modelo            | **definido automaticamente pela celula acima** |
| Arquivos salvos          | `melhor_modelo.pkl`, `label_encoders.pkl`, `features.pkl`, `metadata.json` |

### Principais Insights
- **Ano do modelo** e **marca** sao os fatores com maior impacto no preco
- **Veiculos eletricos e hibridos** tem preco medio significativamente maior
- **Cambio automatico** representa um acrescimo consistente no valor
- A **depreciacao** segue curva nao-linear - carros com +10 anos perdem impacto marginal menor

### Proximos Passos Sugeridos
1. Incorporar dados reais via API FIPE para retreino mensal automatico
2. Adicionar features de texto (descricao do modelo) com NLP
3. Criar um endpoint REST (FastAPI/Flask) para servir previsoes em producao
4. Implementar monitoramento de data drift com Evidently AI
5. Explorar SHAP values para explicabilidade individual de previsoes

---
*Notebook criado com Python 3 | scikit-learn | XGBoost | LightGBM*